# Spark check: can spark read from and write to rustfs?

Connects to the spark connect server in docker and reads the opening book from the gold
bucket, then writes a small test table to silver and reads it back.

In [1]:
from pyspark.sql import SparkSession

# same as in the course notebooks, spark connect runs in docker on port 15002
spark = SparkSession.builder.remote("sc://localhost:15002").appName("spark-check").getOrCreate()

In [2]:
# s3a:// is how spark talks to s3, here that means our rustfs
openings = spark.read.parquet("s3a://gold/openings/openings.parquet")
openings.printSchema()
print(openings.count(), "positions")
openings.select("eco", "name", "plies").show(5, truncate=False)

root
 |-- fen_key: string (nullable = true)
 |-- eco: string (nullable = true)
 |-- eco_family: string (nullable = true)
 |-- name: string (nullable = true)
 |-- moves: string (nullable = true)
 |-- plies: long (nullable = true)
 |-- src: string (nullable = true)
 |-- is_interpolated: boolean (nullable = true)
 |-- is_eco_root: boolean (nullable = true)

15373 positions
+---+-----------------+-----+
|eco|name             |plies|
+---+-----------------+-----+
|A00|Polish Opening   |1    |
|A10|English Opening  |1    |
|A40|Queen's Pawn Game|1    |
|B00|King's Pawn Game |1    |
|A02|Bird Opening     |1    |
+---+-----------------+-----+
only showing top 5 rows


In [3]:
# write a tiny table to silver and read it back, so we know writing works too
test = spark.createDataFrame([("hello", 1), ("rustfs", 2)], ["word", "number"])
test.write.mode("overwrite").parquet("s3a://silver/spark-check")
spark.read.parquet("s3a://silver/spark-check").show()

+------+------+
|  word|number|
+------+------+
|rustfs|     2|
| hello|     1|
+------+------+

